In [2]:
# تحقّق من نوع وحدة معالجة الرسوميات وحجم ذاكرتها (40 أو 80 غيغابايت).
# Check the GPU type and its memory (40 or 80 GB).
!nvidia-smi

Thu Oct 15 22:24:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# ثبّت المكتبات المطلوبة.
# Install the required libraries.
!pip install -q -U transformers peft accelerate datasets

# أزل torchao، لأن نسخة Colab القديمة منها تتعارض مع peft.
# Remove torchao. Colab's old version conflicts with peft.
!pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 42.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [4]:
import os

# اسمح لـ PyTorch بإعادة استخدام الذاكرة المحرَّرة. يجب ضبطه قبل أي استخدام لوحدة الرسوميات.
# Let PyTorch reuse freed memory. Must be set before the GPU is used.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from google.colab import userdata
from huggingface_hub import login, snapshot_download

# سجّل الدخول بالرمز المحفوظ في أسرار Colab (رمز كتابة).
# Log in with the write token stored in Colab secrets.
login(userdata.get("HF_TOKEN"))

# نزّل ملفات النص فقط، دون ملفات .gitkeep وبيانات الضبط الدقيق.
# Download text files only, skipping .gitkeep and the SFT files.
path = snapshot_download(
    "MeshariAlsh/arabic1800s-training",
    repo_type="dataset",
    allow_patterns=["splits/**/*.txt"],
)
print(path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--MeshariAlsh--arabic1800s-training/snapshots/8372f04b39c2751b5543cf124a472a07e66b3715


In [5]:
from pathlib import Path

root = Path(path) / "splits"


def read_dir(name):
    """ Arabic: أعِد قاموسًا {معرّف الكتاب: النص} لكل ملف في المجلد splits/<name>/.
        English: Return {book_id: text} for every file in splits/<name>/.
    """
    return {f.stem.split("-", 1)[1]: f.read_text(encoding="utf-8")
            for f in sorted((root / name).glob("*.txt"))}


train = read_dir("train")
heldout = read_dir("heldout")

# اطبع عدد الكلمات لكل كتاب. يجب أن يكون الجزء المحجوز نحو 5% من جزء التدريب.
# Print word counts per book. Heldout should be about 5% of train.
for book in train:
    print(f"{book:20} train {len(train[book].split()):>8,} words   heldout {len(heldout[book].split()):>6,} words")

clean-book-001       train   59,579 words   heldout  3,202 words
clean-book-002       train   45,344 words   heldout  2,529 words
clean-book-003       train   97,305 words   heldout  5,201 words
clean-book-004       train  230,715 words   heldout 12,159 words
clean-book-005       train   22,327 words   heldout  1,259 words
clean-book-006       train   96,905 words   heldout  5,306 words
clean-book-008       train   24,416 words   heldout  1,308 words
clean-book-009       train  103,949 words   heldout  5,531 words
clean-magazine-001   train  210,226 words   heldout 11,169 words
clean-newspaper-001  train   91,001 words   heldout  4,821 words


In [6]:
from transformers import AutoTokenizer

# حمّل المقسِّم فقط. هو مشترك بين جميع أحجام Qwen3، وقد قيست خصوبته سابقًا (2.40 رمز لكل كلمة).
# Load the tokenizer only. It is shared by all Qwen3 sizes; fertility was measured before (2.40 tokens per word).
MODEL = "Qwen/Qwen3-8B-Base"
tok = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [7]:
from datasets import Dataset

# طول كل مقطع تدريب بالرموز.
# Length of each training block in tokens.
BLOCK = 2048


def pack(texts):
    """ Arabic: حوّل الكتب إلى رموز، وافصل بينها برمز نهاية النص، ثم قسّمها إلى مقاطع كاملة طول كل منها 2048 رمزًا.
        English: Tokenise the books, separate them with an end-of-text token, and cut into full 2,048-token blocks.
    """
    ids = []
    for text in texts.values():
        ids.extend(tok(text)["input_ids"] + [tok.eos_token_id])
    n_blocks = len(ids) // BLOCK
    return [ids[i * BLOCK:(i + 1) * BLOCK] for i in range(n_blocks)]


# جمِّع التدريب والمحجوز كلًّا على حدة، حتى يبقى النص المحجوز غير مرئي للنموذج.
# Pack train and heldout separately, so the held-out text stays unseen.
train_blocks, held_blocks = pack(train), pack(heldout)

# التسميات هي المدخلات نفسها، والنموذج يزيحها بموضع واحد داخليًا.
# Labels are the inputs themselves; the model shifts them by one internally.
train_ds = Dataset.from_dict({"input_ids": train_blocks, "labels": train_blocks})
held_ds = Dataset.from_dict({"input_ids": held_blocks, "labels": held_blocks})

# المتوقع 1149 مقطعًا للتدريب و60 مقطعًا محجوزًا.
# Expect 1149 train blocks and 60 heldout blocks.
print(len(train_ds), len(held_ds))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (136034 > 131072). Running this sequence through the model will result in indexing errors


1149 60


In [8]:
import math
import torch
from transformers import AutoModelForCausalLM

# حمّل النموذج بدقة bf16 كاملة دون تكميم، فذاكرة A100 تتسع له، والتدريب أسرع وبلا خطأ تكميم.
# Load in full bf16 without quantization. The A100 has room, training is faster, and there is no quantization error.
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cuda")

# المتوقع نحو 16.4 غيغابايت.
# Expect about 16.4 GB.
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

GPU memory used: 16.4 GB


In [9]:
@torch.no_grad()
def heldout_loss(model, ds, batch_size=4):
    """ Arabic: متوسط خسارة التنبؤ بالرمز التالي على جميع المقاطع المحجوزة.
        English: Average next-token loss over all held-out blocks.
    """
    model.eval()
    losses = []
    for i in range(0, len(ds), batch_size):
        batch = torch.tensor(ds[i:i + batch_size]["input_ids"], device=model.device)
        losses.append(model(input_ids=batch, labels=batch).loss.item())
    loss = sum(losses) / len(losses)
    return loss, math.exp(loss)


# هذا هو الرقم الذي يجب أن يتفوق عليه التدريب. لا تقارنه بأرقام النموذج 0.6B.
# This is the number training has to beat. Do not compare it with the 0.6B numbers.
base_loss, base_ppl = heldout_loss(model, held_ds)
print(f"8B baseline held-out loss {base_loss:.3f}   perplexity {base_ppl:.1f}")

8B baseline held-out loss 2.851   perplexity 17.3


In [10]:
PROMPT = "أما بعد، فإن"


def sample(model, prompt, n=150):
    """ Arabic: ولِّد نصًا من النموذج انطلاقًا من بداية معطاة.
        English: Generate text from the model, starting from a given prompt.
    """
    model.eval()
    ids = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=n, do_sample=True, temperature=0.7, top_p=0.9)
    return tok.decode(out[0], skip_special_tokens=True)


# شغّل هذه الخلية مرتين أو ثلاثًا، واحفظ النتائج للمقارنة بعد التدريب.
# Run this cell two or three times and keep the outputs to compare after training.
print(sample(model, PROMPT))

أما بعد، فإن أحب الناس إلى الله عز وجل: من اتقاه في الدنيا، ومن عرفه في الآخرة، ومن رآه في الجنة. قال الله عز وجل: (وَالَّذِينَ يَخْشَوْنَ رَبَّهُم بِالْغَيْبِ وَهُمْ مِنْ السَّاعَةِ مُحْضِرُونَ). وفي الصحيحين عن عبد الله بن عمر رضي الله عنهما: أن رسول الله صلى الله عليه وسلم قال: «من عرفني في الدنيا، فقد عرفني في الآخرة، ومن عرفني


In [11]:
from peft import LoraConfig, get_peft_model

# أضف مصفوفات LoRA الصغيرة، وتبقى أوزان النموذج الأصلية مجمّدة.
# Attach the small LoRA matrices. The base model's weights stay frozen.
lora = LoraConfig(
    r=16,                 # الرتبة: الحجم الداخلي للمصفوفتين A وB / rank: inner size of A and B
    lora_alpha=32,        # يُضرب التحديث في alpha/r = 2 / the update is scaled by alpha/r = 2
    lora_dropout=0.05,    # إسقاط عشوائي بسيط للحد من الحفظ / small dropout against memorising
    task_type="CAUSAL_LM",
    # طبقات الانتباه والطبقات الأمامية معًا، لأن الأسلوب يسكن في كليهما.
    # Attention and feed-forward layers together, since style lives in both.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301


In [12]:
from transformers import Trainer, TrainingArguments, default_data_collator

# المستودع الخاص الذي تُرفع إليه نقاط الحفظ والمحوِّل النهائي.
# Private repo for the checkpoints and the final adapter.
HUB_REPO = "MeshariAlsh/qwen3-8b-arabic1800s-cpt"

args = TrainingArguments(
    output_dir="cpt-8b",
    num_train_epochs=2,                  # حقبتان مع الاحتفاظ بأفضل نقطة حفظ / 2 epochs, keep the best checkpoint
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,       # 2 × 4 = 8 مقاطع لكل تحديث / 8 blocks per update, 144 updates per epoch
    gradient_checkpointing=True,         # احتفظ بمدخلات كل طبقة فقط وأعد حساب الباقي / keep only layer inputs, recompute the rest
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-4,                  # حجم كل خطوة تحديث / size of each update
    lr_scheduler_type="cosine",          # ينخفض المعدل بسلاسة حتى نهاية التدريب / rate decays smoothly to the end
    warmup_steps=10,                     # يرتفع المعدل تدريجيًا في البداية / rate rises gradually at the start
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=36,                       # أربعة تقييمات في كل حقبة / four evaluations per epoch
    per_device_eval_batch_size=4,
    save_strategy="steps",
    save_steps=36,                       # يجب أن يساوي eval_steps / must match eval_steps
    save_total_limit=2,
    load_best_model_at_end=True,         # احتفظ بالنقطة ذات أدنى خسارة محجوزة / keep the lowest held-out loss
    metric_for_best_model="eval_loss",
    # ارفع نقاط الحفظ إلى Hugging Face حتى لا يضيع التدريب إذا انقطعت الجلسة.
    # Push checkpoints to Hugging Face so a disconnect does not lose the run.
    push_to_hub=True,
    hub_model_id=HUB_REPO,
    hub_private_repo=True,
    hub_strategy="checkpoint",
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=held_ds,
                  data_collator=default_data_collator, processing_class=tok)
trainer.train()

# ارفع أفضل محوِّل بوصفه النسخة النهائية في المستودع.
# Push the best adapter as the final version in the repo.
trainer.push_to_hub()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
36,2.672854,2.809430
72,2.569416,2.780736
108,2.656458,2.762137
144,2.534593,2.747358
180,2.461658,2.749793
216,2.444915,2.741401
252,2.450186,2.736442
288,2.452202,2.735763


CommitInfo(commit_url='https://huggingface.co/MeshariAlsh/qwen3-8b-arabic1800s-cpt/commit/6b182602a0b4f29fa578a87961549e8fd74ed678', commit_message='End of training', commit_description='', oid='6b182602a0b4f29fa578a87961549e8fd74ed678', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MeshariAlsh/qwen3-8b-arabic1800s-cpt', endpoint='https://huggingface.co', repo_type='model', repo_id='MeshariAlsh/qwen3-8b-arabic1800s-cpt'), pr_revision=None, pr_num=None)

In [13]:
# قارن بخط الأساس، واكتب عينة جديدة بالبداية نفسها.
# Compare with the baseline and write a new sample with the same prompt.
loss, ppl = heldout_loss(model, held_ds)
print(f"After CPT: held-out loss {loss:.3f}  perplexity {ppl:.1f}   (baseline {base_loss:.3f} / {base_ppl:.1f})")
print(sample(model, PROMPT))

After CPT: held-out loss 2.736  perplexity 15.4   (baseline 2.851 / 17.3)
أما بعد، فإن من فوائد التأريخ أن يعلم العبد من قربه من مولاه، ومن كثرة تذكاره، ومن أين أتى، وكم ملك، ومن أين أتى ملكه، وكم كان قبله، ومن أين أتى، ومن أين كان أول ملكه، ومن أين أتى، فمَن قرأ تاريخ العرب، وقرأ تاريخ بني إسرائيل، وقرأ تاريخ فرعون، وقرأ تاريخ موسى، وقرأ تاريخ ملوك الأقباط، وقرأ تاريخ الأمويين، وقرأ تاريخ العباسية، وقرأ تاريخ العباسيين، وقرأ تاريخ الفاطم
